# LLM-as-a-Judge Validation — ClearView MAMSR Synthetic Data

## Purpose
This notebook validates the quality of LLM-generated synthetic cosmetic reviews used to address class imbalance in the ClearView MAMSR training corpus.

```
Input : data/augmented/LLM_Gen_*.csv   (synthetic reviews per aspect/sentiment)
Output: data/augmented/validation_output/validation_results.csv
        data/augmented/validation_output/validation_report.md
```

## Methodology
Follows the **LLM-as-a-Judge** protocol (Zheng et al., 2023):

- A locally hosted `llama3.1` model (via [Ollama](https://ollama.com)) is instantiated as **3 independent judge personas**:
  - **Judge A** — Strict NLP annotation expert
  - **Judge B** — Senior data quality assessor
  - **Judge C** — Consumer research analyst (lenient)
- Each review is scored on **3 criteria**:
  | Criterion | Description |
  |-----------|-------------|
  | `label_correctness` | Does the review express the claimed sentiment class? |
  | `aspect_relevance` | Does the review discuss the claimed cosmetic aspect? |
  | `naturalness` | Does it read like a genuine customer review? |
- **Majority vote** (≥ 2 of 3 judges) determines the final verdict per criterion.
- **Fleiss' Kappa (κ)** quantifies inter-rater reliability across all judges.

## Files Validated
| File | Aspect | Sentiment |
|------|--------|-----------|
| `LLM_Gen_Packing_Neg_Reviews.csv` | packing | negative |
| `LLM_Gen_Packing_Neu_Reviews.csv` | packing | neutral |
| `LLM_Gen_Price_Neg_Reviews.csv` | price | negative |
| `LLM_Gen_Price_Neu_Reviews.csv` | price | neutral |
| `LLM_Gen_Smell_Neu_Reviews.csv` | smell | neutral |

## Prerequisites
```bash
pip install requests pandas numpy tqdm
ollama pull llama3.1
```
> **Note:** Ollama must be running locally at `http://localhost:11434` before executing this notebook.

## 1. Imports & Settings

All standard library and third-party imports, plus the two core configuration blocks:
- **Ollama settings** — base URL and default model name
- **Domain config** — which CSV files to validate, aspect descriptions, sentiment descriptions, and judge criteria

In [1]:
print("⏳ Starting: Loading imports and configuration...")
import time
_start_time = time.time()

import argparse
import json
import random
import re
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

# ─── Ollama Settings ──────────────────────────────────────────────────────────
OLLAMA_BASE   = "http://localhost:11434"
DEFAULT_MODEL = "llama3.1"   # change to 'mistral' or 'gemma2' if preferred

# ─── Runtime Config ───────────────────────────────────────────────────────────
# Adjust these before running the notebook
DATA_DIR   = Path("../data/augmented")           # folder containing LLM_Gen_*.csv files
OUTPUT_DIR = Path("../data/augmented/validation_output") # where results will be saved
SAMPLE_N   = 30                            # number of reviews sampled per CSV file
MODEL      = DEFAULT_MODEL

print(f"  Data dir   : {DATA_DIR.resolve()}")
print(f"  Output dir : {OUTPUT_DIR.resolve()}")
print(f"  Sample N   : {SAMPLE_N} per file")
print(f"  Model      : {MODEL}")

print(f"\n🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading imports and configuration.")

⏳ Starting: Loading imports and configuration...
  Data dir   : C:\Users\lucif\Desktop\Clearview\ml-research\data\augmented
  Output dir : C:\Users\lucif\Desktop\Clearview\ml-research\data\augmented\validation_output
  Sample N   : 30 per file
  Model      : llama3.1

🕒 Done in 0.97s
✅ Completed: Loading imports and configuration.


## 2. Domain Configuration

Defines:
- **`SYNTHETIC_FILES`** — maps each CSV filename to its `(aspect, sentiment)` ground-truth labels
- **`ASPECT_DESCRIPTIONS`** — plain-English descriptions of each aspect, injected into judge prompts
- **`SENTIMENT_DESCRIPTIONS`** — describes what each sentiment class should express
- **`CRITERIA`** — the three quality dimensions scored by every judge

In [2]:
print("⏳ Starting: Setting domain configuration...")
_start_time = time.time()

SYNTHETIC_FILES = {
    "LLM_Gen_Packing_Neg_Reviews.csv":  ("packing",  "negative"),
    "LLM_Gen_Packing_Neu_Reviews.csv":  ("packing",  "neutral"),
    "LLM_Gen_Price_Neg_Reviews.csv":    ("price",    "negative"),
    "LLM_Gen_Price_Neu_Reviews.csv":    ("price",    "neutral"),
    "LLM_Gen_Smell_Neu_Reviews.csv":    ("smell",    "neutral"),
}

ASPECT_DESCRIPTIONS = {
    "packing":      "packaging, box, bottle, container, seal, wrapper",
    "price":        "cost, price, value for money, affordability",
    "smell":        "scent, fragrance, odour, aroma of the product",
    "texture":      "feel, consistency, thickness, spreadability on skin",
    "colour":       "colour, shade, pigmentation, hue",
    "shipping":     "delivery, courier, arrival condition",
    "stayingpower": "how long the product lasts, wear time, longevity",
}

SENTIMENT_DESCRIPTIONS = {
    "positive": "satisfaction, praise, or approval",
    "neutral":  "factual, balanced, or neither clearly positive nor negative",
    "negative": "dissatisfaction, complaint, or criticism",
}

CRITERIA = ["label_correctness", "aspect_relevance", "naturalness"]

print(f"  Files to validate : {len(SYNTHETIC_FILES)}")
print(f"  Aspects covered   : {sorted(set(v[0] for v in SYNTHETIC_FILES.values()))}")
print(f"  Criteria          : {CRITERIA}")

print(f"\n🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Setting domain configuration.")

⏳ Starting: Setting domain configuration...
  Files to validate : 5
  Aspects covered   : ['packing', 'price', 'smell']
  Criteria          : ['label_correctness', 'aspect_relevance', 'naturalness']

🕒 Done in 0.00s
✅ Completed: Setting domain configuration.


## 3. Judge Personas & Prompt Template

Three separate system prompts instantiate different annotation personas within the same LLM:

| Persona | Role | Bias |
|---------|------|------|
| **Judge A** | Strict NLP annotation expert | Marks ambiguous reviews as FAIL |
| **Judge B** | Senior data quality assessor | Applies moderate, consistent standards |
| **Judge C** | Consumer research analyst | Gives benefit of the doubt |

The judge prompt injects the review text, claimed aspect, and claimed sentiment, then asks for a structured JSON verdict.

In [3]:
print("⏳ Starting: Defining judge personas and prompt template...")
_start_time = time.time()

JUDGE_PERSONAS = [
    {
        "name": "Judge_A",
        "system": (
            "You are a strict NLP annotation expert specialising in Aspect-Based Sentiment Analysis. "
            "You evaluate synthetic cosmetic reviews with high standards. "
            "If a review is ambiguous, mark it FAIL. "
            "You MUST respond ONLY with a valid JSON object. No explanation, no extra text."
        ),
    },
    {
        "name": "Judge_B",
        "system": (
            "You are a senior data quality assessor for an NLP research team. "
            "Validate whether machine-generated cosmetic reviews are suitable for training a sentiment classifier. "
            "Apply consistent, moderate standards. "
            "You MUST respond ONLY with a valid JSON object. No explanation, no extra text."
        ),
    },
    {
        "name": "Judge_C",
        "system": (
            "You are a consumer research analyst experienced with cosmetic product reviews. "
            "Assess whether auto-generated reviews are realistic and correctly labelled. "
            "Give benefit of the doubt when intent is clear. "
            "You MUST respond ONLY with a valid JSON object. No explanation, no extra text."
        ),
    },
]

JUDGE_PROMPT = """Evaluate this synthetic cosmetic product review.

REVIEW TEXT:
\"\"\"{review_text}\"\"\"

CLAIMED ASPECT: {aspect}
(Review should discuss: {aspect_description})

CLAIMED SENTIMENT: {sentiment}
(Review should express: {sentiment_description})

Evaluate on exactly three criteria. Respond ONLY with this JSON, nothing else:

{{
  "label_correctness": {{"verdict": "PASS", "reason": "one sentence why"}},
  "aspect_relevance":  {{"verdict": "PASS", "reason": "one sentence why"}},
  "naturalness":       {{"verdict": "PASS", "reason": "one sentence why"}}
}}

Rules:
- label_correctness = PASS if review clearly expresses {sentiment} sentiment toward {aspect}
- aspect_relevance  = PASS if review actually discusses {aspect} content
- naturalness       = PASS if review reads like a genuine human-written cosmetic review
- Use FAIL if a criterion is not met
- Output ONLY the JSON object, no other text before or after
"""

print(f"  Judges defined : {[j['name'] for j in JUDGE_PERSONAS]}")

print(f"\n🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining judge personas and prompt template.")

⏳ Starting: Defining judge personas and prompt template...
  Judges defined : ['Judge_A', 'Judge_B', 'Judge_C']

🕒 Done in 0.00s
✅ Completed: Defining judge personas and prompt template.


## 4. Ollama API Functions (with GPU Detection)

- **`check_ollama(model)`** — verifies that Ollama is running and the requested model is downloaded. **Reports whether the model is loaded on GPU or CPU.**
- **`call_ollama(system_prompt, user_prompt, model)`** — sends a low-temperature chat request (temperature=0.1) for consistent JSON output.
- **`call_judge(judge, review_text, aspect, sentiment, model)`** — wraps `call_ollama` with JSON parsing and 3-attempt retry logic. Strips markdown code fences that some models add around JSON.

In [4]:
print("⏳ Starting: Defining Ollama API functions...")
_start_time = time.time()

def check_ollama(model: str):
    """Check Ollama is running, model exists, and determine hardware (GPU/CPU)."""
    try:
        r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models", [])]
    except requests.exceptions.ConnectionError:
        print("\n[ERROR] Cannot connect to Ollama at http://localhost:11434")
        print("        Make sure Ollama is installed and running.")
        raise RuntimeError("Ollama not reachable")

    matched = [m for m in models if model.split(":")[0] in m]
    if not matched:
        print(f"\n[ERROR] Model '{model}' not found locally.")
        raise RuntimeError(f"Model '{model}' not available")

    # Attempt to detect hardware via /api/ps (wait for model to load if needed)
    hw_info = "[Detecting...]"
    import time as t_lib
    for attempt in range(3):
        try:
            # Ping /api/generate with a tiny prompt to ensure the model sits in VRAM
            requests.post(f"{OLLAMA_BASE}/api/generate", json={"model": matched[0], "prompt": "hi", "stream": False}, timeout=10)
            
            r_ps = requests.get(f"{OLLAMA_BASE}/api/ps", timeout=5)
            ps_data = r_ps.json().get("models", [])
            model_ps = next((m for m in ps_data if matched[0] in m["name"]), None)
            
            if model_ps:
                proc = model_ps.get("processor", "Unknown")
                if proc == "Unknown":
                    t_lib.sleep(2) # wait for hardware enumeration
                    continue
                hw_info = f"🚀 Running on {proc}"
                break
            else:
                hw_info = "🎮 Hardware detection: Ollama will choose best (typically GPU if available)"
                t_lib.sleep(1)
        except:
            hw_info = "🎮 Hardware status: Native (check nvidia-smi for VRAM usage)"

    print(f"  ✅ Ollama OK — Model: {matched[0]}")
    print(f"  {hw_info}")
    return matched[0]


def call_ollama(system_prompt: str, user_prompt: str, model: str) -> str:
    """Send a chat request to local Ollama and return the response text."""
    payload = {
        "model":  model,
        "stream": False,
        "options": {
            "temperature": 0.1,  # Low temperature → consistent JSON output
            "top_p": 0.9,
        },
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
    }
    response = requests.post(
        f"{OLLAMA_BASE}/api/chat",
        json=payload,
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["message"]["content"]


def call_judge(judge: dict, review_text: str, aspect: str,
               sentiment: str, model: str) -> dict | None:
    """Call one judge persona and return a parsed verdict dict. Retries 3×."""
    prompt = JUDGE_PROMPT.format(
        review_text=review_text[:500],
        aspect=aspect,
        aspect_description=ASPECT_DESCRIPTIONS.get(aspect, aspect),
        sentiment=sentiment,
        sentiment_description=SENTIMENT_DESCRIPTIONS.get(sentiment, sentiment),
    )

    for attempt in range(3):
        try:
            raw = call_ollama(judge["system"], prompt, model).strip()
            raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.MULTILINE)
            raw = re.sub(r"\s*```$",          "", raw, flags=re.MULTILINE)
            match = re.search(r"\{.*\}", raw, re.DOTALL)
            if match:
                raw = match.group(0)
            return json.loads(raw)
        except json.JSONDecodeError:
            t_lib.sleep(1)
        except Exception as e:
            t_lib.sleep(2)
    return None


print(f"\n🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining Ollama API functions.")

⏳ Starting: Defining Ollama API functions...

🕒 Done in 0.00s
✅ Completed: Defining Ollama API functions.


## 5. Statistics — Fleiss' Kappa

**Fleiss' Kappa (κ)** measures inter-rater agreement for categorical ratings across multiple raters.

Here it is applied as a **binary agreement** measure (PASS=1 / FAIL=0) across the 3 judge personas:

$$\kappa = \frac{\bar{P} - P_e}{1 - P_e}$$

| κ range | Interpretation |
|---------|----------------|
| < 0.20 | Slight |
| 0.20–0.40 | Fair |
| 0.40–0.60 | Moderate |
| 0.60–0.80 | Substantial |
| > 0.80 | Almost Perfect |

> *Source: Landis & Koch (1977)*

In [5]:
print("⏳ Starting: Defining Fleiss' Kappa functions...")
_start_time = time.time()

def fleiss_kappa(ratings_matrix: np.ndarray) -> float:
    """
    Fleiss' Kappa for multi-rater binary agreement.
    """
    n_items, n_categories = ratings_matrix.shape
    n_raters = int(ratings_matrix[0].sum())
    if n_raters < 2:
        return float("nan")

    p_j   = ratings_matrix.sum(axis=0) / (n_items * n_raters)
    P_e   = (p_j ** 2).sum()
    P_i   = ((ratings_matrix ** 2).sum(axis=1) - n_raters) / (n_raters * (n_raters - 1))
    P_bar = P_i.mean()

    if abs(1 - P_e) < 1e-9:
        return 1.0
    return float((P_bar - P_e) / (1 - P_e))


def interpret_kappa(k: float) -> str:
    """Return a human-readable label for a Fleiss' Kappa value."""
    if np.isnan(k): return "N/A"
    if k < 0:       return "Poor"
    if k < 0.20:    return "Slight"
    if k < 0.40:    return "Fair"
    if k < 0.60:    return "Moderate"
    if k < 0.80:    return "Substantial"
    return "Almost Perfect"


print(f"\n🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining Fleiss' Kappa functions.")

⏳ Starting: Defining Fleiss' Kappa functions...

🕒 Done in 0.00s
✅ Completed: Defining Fleiss' Kappa functions.


## 6. Step 1 — Check Ollama & Load Samples

- Verifies Ollama hardware support.
- Loads `SAMPLE_N` random reviews from each synthetic CSV.

In [6]:
print("⏳ Starting: Checking Ollama and loading samples...")
_start_time = time.time()

random.seed(42)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL = check_ollama(MODEL)

def load_samples(data_dir: Path, sample_n: int) -> list:
    samples = []
    for filename, (aspect, sentiment) in SYNTHETIC_FILES.items():
        filepath = data_dir / filename
        if not filepath.exists():
            print(f"  [WARN] Not found: {filepath}")
            continue
        df = pd.read_csv(filepath)
        text_col = next((c for c in ["review", "text", "Review", "Text", "review_text", "content"] if c in df.columns), df.columns[0])
        texts   = df[text_col].dropna().astype(str).tolist()
        n       = min(sample_n, len(texts))
        sampled = random.sample(texts, n)
        for t in sampled:
            samples.append({"file": filename, "aspect": aspect, "sentiment": sentiment, "review_text": t})
        print(f"  Loaded {n}/{len(texts)} samples from {filename}")
    return samples

samples = load_samples(DATA_DIR, SAMPLE_N)
total   = len(samples)
print(f"\n  Total samples to validate: {total}")
print(f"\n🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Checking Ollama and loading samples.")

⏳ Starting: Checking Ollama and loading samples...
  ✅ Ollama OK — Model: llama3.1:latest
  🎮 Hardware status: Native (check nvidia-smi for VRAM usage)
  Loaded 30/192 samples from LLM_Gen_Packing_Neg_Reviews.csv
  Loaded 30/196 samples from LLM_Gen_Packing_Neu_Reviews.csv
  Loaded 30/176 samples from LLM_Gen_Price_Neg_Reviews.csv
  Loaded 30/193 samples from LLM_Gen_Price_Neu_Reviews.csv
  Loaded 30/166 samples from LLM_Gen_Smell_Neu_Reviews.csv

  Total samples to validate: 150

🕒 Done in 29.39s
✅ Completed: Checking Ollama and loading samples.


## 7. Step 2 — Run Judge Validation (with ETA Progress Bar)

This cell uses a `tqdm` progress bar to track:
- Number of samples processed
- Percentage completed
- Elapsed time
- **Estimated Time Remaining (ETA)**

The console output below also prints the individual judge verdicts for each sample in real-time.

In [7]:
print(f"⏳ Starting: Running {total} validation samples (3 judges per sample)...\n")
_start_eval_time = time.time()
results = []

# Main evaluation loop with progress bar
pbar = tqdm(samples, desc="Validating Samples", unit="sample")
for i, sample in enumerate(pbar):
    # Detailed real-time logging below the progress bar
    print(f"\rSample [{i+1}/{total}] | {sample['aspect']:10s} | {sample['sentiment']:8s} | {sample['review_text'][:40]}...", end="\n")
    
    row = {
        "sample_id":   i + 1,
        "file":        sample["file"],
        "aspect":      sample["aspect"],
        "sentiment":   sample["sentiment"],
        "review_text": sample["review_text"],
    }
    judge_verdicts = {}

    for judge in JUDGE_PERSONAS:
        verdict = call_judge(judge, sample["review_text"], sample["aspect"], sample["sentiment"], MODEL)
        if verdict is None:
            judge_verdicts[judge["name"]] = {c: None for c in CRITERIA}
            for c in CRITERIA:
                row[f"{judge['name']}_{c}"] = "ERROR"
            continue

        parsed = {}
        for c in CRITERIA:
            v = verdict.get(c, {}).get("verdict", "FAIL")
            parsed[c] = 1 if str(v).upper() == "PASS" else 0
            row[f"{judge['name']}_{c}"] = str(v).upper()
            row[f"{judge['name']}_{c}_reason"] = verdict.get(c, {}).get("reason", "")
        judge_verdicts[judge["name"]] = parsed

        icons = " ".join(f"[{'✓' if parsed[c] else '✗'} {c[0].upper()}]" for c in CRITERIA)
        print(f"   └─ {judge['name']:7s}: {icons}")

    # Majority vote
    for c in CRITERIA:
        votes = [judge_verdicts[j["name"]][c] for j in JUDGE_PERSONAS if judge_verdicts.get(j["name"], {}).get(c) is not None]
        row[f"majority_{c}"] = (1 if sum(votes) >= 2 else 0) if votes else None
    
    row["overall_valid"] = int(all(row.get(f"majority_{c}", 0) == 1 for c in CRITERIA))
    results.append(row)

total_duration = time.time() - _start_eval_time
print(f"\n🕒 Total Evaluation Time: {timedelta(seconds=int(total_duration))}")
print("✅ Completed: Running judge validation.")

⏳ Starting: Running 150 validation samples (3 judges per sample)...



Validating Samples:   0%|          | 0/150 [00:00<?, ?sample/s]

Sample [1/150] | packing    | negative | Packaging was not secure, the box opened...
   └─ Judge_A: [✓ L] [✓ A] [✓ N]
   └─ Judge_B: [✓ L] [✓ A] [✗ N]
   └─ Judge_C: [✓ L] [✓ A] [✓ N]
Sample [2/150] | packing    | negative | Bad packaging, the box was already dente...
   └─ Judge_A: [✓ L] [✓ A] [✗ N]
   └─ Judge_B: [✓ L] [✓ A] [✗ N]
   └─ Judge_C: [✓ L] [✓ A] [✓ N]
Sample [3/150] | packing    | negative | Bad packaging, the outer box was damaged...
   └─ Judge_A: [✓ L] [✓ A] [✓ N]
   └─ Judge_B: [✓ L] [✗ A] [✓ N]
   └─ Judge_C: [✓ L] [✓ A] [✓ N]
Sample [4/150] | packing    | negative | Shipping was quick, but the packaging wa...
   └─ Judge_A: [✓ L] [✓ A] [✓ N]
   └─ Judge_B: [✓ L] [✗ A] [✓ N]
   └─ Judge_C: [✓ L] [✓ A] [✓ N]
Sample [5/150] | packing    | negative | Good staying power, even after eating. B...
   └─ Judge_A: [✓ L] [✓ A] [✓ N]
   └─ Judge_B: [✓ L] [✗ A] [✓ N]
   └─ Judge_C: [✓ L] [✗ A] [✓ N]
Sample [6/150] | packing    | negative | Fast delivery but the courier bag was r

## 8. Step 3 — Compute Statistics & Save CSV

In [8]:
print("⏳ Starting: Computing statistics and saving results CSV...")
df_results = pd.DataFrame(results)
csv_path = OUTPUT_DIR / "validation_results.csv"
df_results.to_csv(csv_path, index=False)
print(f"  💾 Saved raw results → {csv_path}")

n_total = len(df_results)
if n_total == 0:
    print("❌ No results found.")
    n_valid = 0; pct = 0
else:
    n_valid = int(df_results["overall_valid"].sum())
    pct = 100 * n_valid / n_total
    criterion_rates = {c: 100 * df_results[f"majority_{c}"].mean() for c in CRITERIA}
    per_file = df_results.groupby("file")["overall_valid"].agg(["sum","count"])
    per_file["pct"] = 100 * per_file["sum"] / per_file["count"]
    per_aspect = df_results.groupby("aspect")["overall_valid"].agg(["sum","count"])
    per_aspect["pct"] = 100 * per_aspect["sum"] / per_aspect["count"]
    
    kappas = {}
    for c in CRITERIA:
        rows = [[sum(1 for j in JUDGE_PERSONAS if str(r.get(f"{j['name']}_{c}", "FAIL")).upper() == "PASS"), 3 - sum(1 for j in JUDGE_PERSONAS if str(r.get(f"{j['name']}_{c}", "FAIL")).upper() == "PASS")] for _, r in df_results.iterrows()]
        kappas[c] = fleiss_kappa(np.array(rows))
print("✅ Completed: Computing statistics.")

⏳ Starting: Computing statistics and saving results CSV...
  💾 Saved raw results → ..\data\augmented\validation_output\validation_results.csv
✅ Completed: Computing statistics.


## 9. Results Summary

In [9]:
print(f"\n{'='*60}\nRESULTS\n{'='*60}")
print(f"  Samples evaluated : {n_total}")
print(f"  Overall valid     : {n_valid}/{n_total} ({pct:.1f}%)\n")
for c in CRITERIA:
    print(f"  {c.replace('_',' ').title():22s}: {criterion_rates.get(c,0):5.1f}% | κ = {kappas.get(c,0):.3f} ({interpret_kappa(kappas.get(c,0))})")
print(f"\n  Per-file:\n" + "\n".join(f"    {fn:45s}: {r['pct']:.1f}%" for fn, r in per_file.iterrows()))
print(f"{'='*60}\n")


RESULTS
  Samples evaluated : 150
  Overall valid     : 110/150 (73.3%)

  Label Correctness     : 100.0% | κ = 1.000 (Almost Perfect)
  Aspect Relevance      :  84.0% | κ = 0.514 (Moderate)
  Naturalness           :  89.3% | κ = 0.317 (Fair)

  Per-file:
    LLM_Gen_Packing_Neg_Reviews.csv              : 56.7%
    LLM_Gen_Packing_Neu_Reviews.csv              : 56.7%
    LLM_Gen_Price_Neg_Reviews.csv                : 86.7%
    LLM_Gen_Price_Neu_Reviews.csv                : 73.3%
    LLM_Gen_Smell_Neu_Reviews.csv                : 93.3%



## 10. Interactive Exploration

In [10]:
if n_total > 0:
    display(df_results[["sample_id", "aspect", "sentiment", "overall_valid"]].head(10))
    display(pd.DataFrame([{"Criterion": c.title(), "κ": f"{kappas[c]:.3f}"} for c in CRITERIA]))

,sample_id,aspect,sentiment,overall_valid
0,1,packing,negative,1
1,2,packing,negative,0
2,3,packing,negative,1
3,4,packing,negative,1
4,5,packing,negative,0
5,6,packing,negative,1
6,7,packing,negative,1
7,8,packing,negative,1
8,9,packing,negative,0
9,10,packing,negative,0


,Criterion,κ
0,Label_Correctness,1.000
1,Aspect_Relevance,0.514
2,Naturalness,0.317


## 11. Generate Thesis Report

In [ ]:
def build_report():
    now = datetime.now().strftime("%Y-%m-%d %H:%M")
    lines = [f"# Validation Report\nGenerated: {now}\nModel: {MODEL}\nSamples: {n_total}\nValid: {pct:.1f}%\n"]
    for c in CRITERIA: lines.append(f"- {c.title()} Pass Rate: {criterion_rates[c]:.1f}% (κ={kappas[c]:.3f})")
    return "\n".join(lines)

if n_total > 0:
    report = build_report()
    (OUTPUT_DIR / "validation_report.md").write_text(report)
    print("📄 Report saved!")

UnicodeEncodeError: 'charmap' codec can't encode character '\u03ba' in position 143: character maps to <undefined>